# Notebook 08: Neuro-Symbolic Hybrid Agent & Ablation Study
Investigating the Neuro-Symbolic Hybrid Solver combining symbolic hard constraint filtering, Transformer neural log-likelihood, and Information-Theoretic Entropy gain:

$$\text{FinalScore}(w) = \alpha \cdot \text{NeuralScore}(w) + \beta \cdot \text{EntropyScore}(w) + \gamma \cdot \text{FrequencyScore}(w)$$

In [ ]:
import sys
import os
import zipfile

# Auto-extract zip file if Kaggle uploaded dataset as a raw zip
if os.path.exists("/kaggle/input"):
    for root_dir, dirs, files in os.walk("/kaggle/input"):
        for file in files:
            if file.endswith(".zip") and "wordle" in file.lower():
                zip_path = os.path.join(root_dir, file)
                print(f"Extracting uploaded Kaggle dataset: {zip_path} -> /kaggle/working/")
                with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                    zip_ref.extractall("/kaggle/working/")

# Dynamic path resolution for local and Kaggle environments
possible_roots = [
    ".",
    "..",
    "/kaggle/working",
    "/kaggle/working/wordle",
    "/kaggle/working/wordle-ai",
    "/kaggle/input/wordle",
    "/kaggle/input/wordle/wordle",
    "/kaggle/input/wordle/wordle-ai",
    "/kaggle/input/wordle-ai",
]

# Search recursively inside /kaggle/input for any folder containing 'src'
if os.path.exists("/kaggle/input"):
    for root_dir, dirs, files in os.walk("/kaggle/input"):
        if "src" in dirs:
            possible_roots.insert(0, root_dir)

for root in possible_roots:
    if os.path.exists(os.path.join(root, "src")):
        abs_root = os.path.abspath(root)
        if abs_root not in sys.path:
            sys.path.insert(0, abs_root)
        print(f"Found 'src' package at: {abs_root}")
        break

if os.path.exists("/kaggle/input"):
    print("Running on Kaggle GPU/CPU Environment!")
else:
    print("Running on Local Environment!")

from src.wordle.dictionary import WordleDictionary
from src.solvers.hybrid_solver import HybridSolver
from src.evaluation.benchmark import BenchmarkRunner

dict_mgr = WordleDictionary()
runner = BenchmarkRunner(dict_mgr)
test_set = dict_mgr.test_secrets[:25]

# Ablation A: Full Hybrid
hybrid_full = HybridSolver(dict_mgr, alpha=0.5, beta=0.4, gamma=0.1, enforce_constraints=True)
m_full = runner.evaluate_solver(hybrid_full, test_secrets=test_set)

# Ablation B: Without Entropy (beta=0)
hybrid_no_entropy = HybridSolver(dict_mgr, alpha=0.9, beta=0.0, gamma=0.1, enforce_constraints=True)
m_no_entropy = runner.evaluate_solver(hybrid_no_entropy, test_secrets=test_set)

print(f"Full Hybrid Win Rate: {m_full.win_rate:.1f}% | Avg Guesses: {m_full.avg_guesses:.2f}")
print(f"Hybrid (No Entropy) Win Rate: {m_no_entropy.win_rate:.1f}% | Avg Guesses: {m_no_entropy.avg_guesses:.2f}")